In [2]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
# Inserisco qui il link dell'API dell'ISTAT
url_api = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"

headers = {'Accept': 'text/csv'}

# Faccio la richiesta al server per scaricare i dati
print("Download dei dati in corso...")
risposta = requests.get(url_api, headers=headers)

# Controllo che il download sia andato a buon fine (il codice 200 significa OK)
if risposta.status_code == 200:
    print("Download completato! Salvo il file CSV...")
    
    # Salvo il contenuto in CSV
    with open("incidenti_istat.csv", 'wb') as file:
        file.write(risposta.content)
        
    # Carico i dati in Pandas per poterli analizzare
    df_incidenti = pd.read_csv("incidenti_istat.csv")
    
    # Guardo le prime 5 righe per assicurarmi che sia tutto a posto
    display(df_incidenti.head())

else:
    print("Errore durante il download. Codice:", risposta.status_code)

Download dei dati in corso...
Download completato! Salvo il file CSV...


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Ricarico i dati del CSV
df_incidenti = pd.read_csv("incidenti_istat.csv")

display(df_incidenti.info())

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

None

In [4]:
# Pulizia colonne
colonne_da_tenere = ['REF_AREA', 'DATA_TYPE', 'RESULT', 'TIME_PERIOD', 'OBS_VALUE']
df_pulito = df_incidenti[colonne_da_tenere].copy()

# Esplorazione dei valori
print("Valori unici in DATA_TYPE:", df_pulito['DATA_TYPE'].unique())
print("Valori unici in RESULT:", df_pulito['RESULT'].unique())

Valori unici in DATA_TYPE: <StringArray>
['KILLINJ', 'ROADACC']
Length: 2, dtype: str
Valori unici in RESULT: <StringArray>
['F', 'M', '9']
Length: 3, dtype: str


In [5]:
# Filtro e tengo solo gli incidenti stradali ('ROADACC') e il totale ('9')
df_filtrato = df_pulito[(df_pulito['DATA_TYPE'] == 'ROADACC') & (df_pulito['RESULT'] == '9')].copy()

# Rimuovo le colonne che ora contengono valori tutti uguali
df_filtrato.drop(columns=['DATA_TYPE', 'RESULT'], inplace=True)

# Rinomino le colonne per renderle più chiare
df_filtrato.rename(columns={
    'REF_AREA': 'codice_comune',
    'TIME_PERIOD': 'anno',
    'OBS_VALUE': 'totale_incidenti'
}, inplace=True)

# 4. Salvo il dataset pulito in un nuovo file CSV
df_filtrato.to_csv("incidenti_puliti.csv", index=False)

print("Pulizia completata! Ecco il dataset pronto per l'analisi:")
display(df_filtrato.head())

Pulizia completata! Ecco il dataset pronto per l'analisi:


,codice_comune,anno,totale_incidenti
48,1001,2001,5
49,1001,2002,5
50,1001,2003,4
51,1001,2004,9
52,1001,2005,2


In [6]:
# Carico il CSV situas.istat
df_popolazione = pd.read_csv("Comuni.csv", sep=';') 

# 2. Vedo le colonne per capire come si chiamano
print("Colonne del CSV situas.istat:")
print(df_popolazione.columns)

# 3. Vedo le prime righe
display(df_popolazione.head())

Colonne del CSV situas.istat:
Index(['Codice Ripartizione geografica', 'Codice Regione',
       'Codice Provincia (Storico)', 'Codice Provincia/Uts',
       'Codice Comune (alfanumerico)', 'Codice Comune (numerico)', 'Comune',
       'Comune (dizione straniera)', 'Sigla automobilistica',
       'Capoluogo di Provincia/Uts', 'Capoluogo di Regione',
       'Popolazione legale', 'Anno Censimento', 'Superficie (Kmq)',
       'Anno (Superficie)', 'Popolazione residente',
       'Anno (Popolazione residente)'],
      dtype='str')


,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,1,1,201,1001,1001,Agliè,NaN,TO,0,0,2644,2011,"13,1462",2020,2545,2020
1,1,1,1,201,1002,1002,Airasca,NaN,TO,0,0,3819,2011,"15,7393",2020,3633,2020
2,1,1,1,201,1003,1003,Ala di Stura,NaN,TO,0,0,462,2011,"46,3315",2020,459,2020
3,1,1,1,201,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1791,2011,"11,7314",2020,1638,2020
4,1,1,1,201,1006,1006,Almese,NaN,TO,0,0,6303,2011,"17,8756",2020,6355,2020


In [7]:
# Rendo i codici numeri interi per poterli confrontare correttamente
df_filtrato['codice_comune'] = df_filtrato['codice_comune'].astype(int)
df_popolazione['Codice Comune (numerico)'] = df_popolazione['Codice Comune (numerico)'].astype(int)

# Isolo solo le colonne che mi servono dalla tabella della popolazione (creo un piccolo dataframe temporaneo più facile da leggere)
colonne_utili = ['Codice Comune (numerico)', 'Popolazione residente', 'Comune']
df_pop_ridotto = df_popolazione[colonne_utili]

# Eseguo il merge unendo la tabella incidenti con la tabellina della popolazione
df_finale = pd.merge(df_filtrato, df_pop_ridotto, left_on='codice_comune', right_on='Codice Comune (numerico)', how='inner')

# Elimino la colonna doppia che si è creata col merge usando il drop
df_finale = df_finale.drop(columns=['Codice Comune (numerico)'])

# Creo la nuova colonna per il tasso pro-capite (incidenti / popolazione * 1000 più leggibile)
df_finale['incidenti_pro_capite'] = (df_finale['totale_incidenti'] / df_finale['Popolazione residente']) * 1000

# Guardo le prime 5 righe per assicurarmi che sia tutto ok
print("Dataset finale dopo la pulizia e il calcolo:")
display(df_finale.head())

# Esporto il CSV
df_finale.to_csv("dataset_finale_analizzato.csv", index=False)

Dataset finale dopo la pulizia e il calcolo:


,codice_comune,anno,totale_incidenti,Popolazione residente,Comune,incidenti_pro_capite
0,1001,2001,5,2545,Agliè,1.964637
1,1001,2002,5,2545,Agliè,1.964637
2,1001,2003,4,2545,Agliè,1.571709
3,1001,2004,9,2545,Agliè,3.536346
4,1001,2005,2,2545,Agliè,0.785855
